# Phase 10 — chronological demand forecasting
Target: observed `num_orders` per center-meal-week. Fixed-origin forecast: fit weeks 1–135, recursively predict 136–145 without observing any holdout labels. All three models face the same ten-week horizon. After selection, refit through week 145 and recursively predict 146–155. No random split or tuning on holdout. Phase 11 is outside scope.

The installed scikit-learn skill and its pipeline reference guide train-only imputation/scaling/encoding, unknown-category handling, MAE, and seeded Random Forest. See `docs/phase10_progress.md` for actual use and limitations.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.machine_learning.forecast import run
results = run(ROOT)

Evaluating Previous-week baseline


Evaluating Linear Regression


Evaluating Random Forest


Refitting selected model: Random Forest


## Features and information contract
Known meal/center metadata and planned weekly prices/promotions are assumed available at the origin for every forecast week. These are scenario forecasts if the marketing plan changes. IDs are preserved for alignment and excluded as predictors; center/meal/location codes are categorical. Week is a trend index, not calendar seasonality.

Exact prior-week demand and mean/count over the preceding four numbered weeks use strictly earlier records. Missing weeks remain absent. Training features use earlier actuals; recursive holdout/test features use earlier predictions after the origin. No full-period target aggregates. Numeric median imputation and scaling, categorical one-hot encoding, and all fitting use fitting rows only. Baseline missing lag uses the fitting-window global median. Discount `(base-checkout)/base` remains negative when checkout exceeds base; invalid/nonpositive denominators become missing for training-only imputation.

Random Forest settings are fixed in code (40 trees, depth 14, bootstrap sample fraction 0.35, feature fraction 0.5, leaf minimum 5, seed 42); no holdout-driven tuning. Negative Linear Regression outputs are clipped to zero for both evaluation and deployment, with counts recorded.

In [2]:
print(json.dumps({k:results[k] for k in ['design','fit_weeks','holdout_weeks','fit_rows','holdout_rows','fit_orders','holdout_orders','features','joins']},indent=2))

{
  "design": "fixed-origin recursive ten-week forecast; no holdout actual updates",
  "fit_weeks": [
    1,
    135
  ],
  "holdout_weeks": [
    136,
    145
  ],
  "fit_rows": 423727,
  "holdout_rows": 32821,
  "fit_orders": 112097205,
  "holdout_orders": 7460280,
  "features": [
    "center_id",
    "meal_id",
    "category",
    "cuisine",
    "center_type",
    "city_code",
    "region_code",
    "week",
    "op_area",
    "checkout_price",
    "base_price",
    "discount_rate",
    "emailer_for_promotion",
    "homepage_featured",
    "lag_1",
    "prior_4_mean",
    "prior_4_count"
  ],
  "joins": [
    {
      "lookup": "meal_info.csv",
      "before": 32573,
      "after": 32573,
      "unmatched": 0,
      "duplicate_expansion": 0,
      "orders_total_before": "not applicable",
      "orders_total_after": "not applicable",
      "unique_ids_before": 32573,
      "unique_ids_after": 32573,
      "lookup_unique_keys": 51,
      "lookup_rows": 51,
      "row_order_preserved": t

In [3]:
import pandas as pd
comparison = pd.DataFrame(results['metrics']).T
print(comparison.to_string())
print('Selected by lowest holdout MAE:',results['winner'])
print(pd.DataFrame(results['weekly_mae']).to_string(index=False))

                               mae  missing_previous_week  negative_predictions_clipped  improvement_pct_vs_baseline
Previous-week baseline  119.613053                 1576.0                           0.0                     0.000000
Linear Regression       115.689068                 1576.0                        5947.0                     3.280565
Random Forest            80.482608                 1576.0                           0.0                    32.714193
Selected by lowest holdout MAE: Random Forest
 week  rows  Previous-week baseline  Linear Regression  Random Forest
  136  3273               85.457379         102.142255      67.585272
  137  3283              108.993908         111.619240      69.936600
  138  3278              125.325808         111.033975      74.429467
  139  3279              118.906069         110.863033      75.605138
  140  3332              121.323529         126.278140      84.084560
  141  3263              128.760650         116.413125      91.037

## Interpretation and operational use
MAE is the average absolute error in orders per observed center-meal-week; lower is better. The percentage improvement is `100 × (baseline MAE − model MAE) / baseline MAE`. Negative values mean deterioration. Weekly MAE shows where errors remain. Only one historical ten-week period was evaluated; selection on it makes its winning score optimistic as a future guarantee.

The next cell shows standardized numeric linear coefficients and one-hot category coefficients, plus forest impurity importance. These are descriptive model associations, not causal promotion effects. Correlated features share importance; impurity importance favors variables with many potential splits. No tuning follows interpretation.

In [4]:
for name, rows in results['interpretation'].items():
    print(name)
    print(pd.DataFrame(rows).to_string(index=False))

Linear Regression
                       feature       value
           numeric__base_price -236.834741
       numeric__checkout_price  161.468646
categories__category_Rice Bowl  159.355507
      categories__meal_id_2290  155.372258
      categories__meal_id_1885  154.806933
      categories__meal_id_1248 -131.529157
      categories__meal_id_2631 -124.163152
                numeric__lag_1  123.080280
      categories__meal_id_2306 -105.124775
     categories__category_Soup  -90.570424
      categories__center_id_43   89.311469
      categories__meal_id_1993   87.089586
         numeric__prior_4_mean   85.865369
    categories__category_Pasta  -81.779105
        numeric__discount_rate   80.475373
numeric__emailer_for_promotion   78.889518
      categories__meal_id_2707   73.438839
      categories__center_id_53  -57.402599
   categories__category_Extras  -55.587068
     categories__center_id_137   55.551608
Random Forest
                        feature    value
                 numeric

In [5]:
print(json.dumps(results['test_summary'],indent=2))
predictions = pd.read_csv(ROOT/'data/processed/phase10/test_predictions.csv')
raw_test = pd.read_csv(ROOT/'data/raw/test.csv')
assert predictions.id.tolist() == raw_test.id.tolist()
print(predictions.head().to_string(index=False))

{
  "rows": 32573,
  "weeks": [
    146,
    155
  ],
  "minimum": 17.5214286844975,
  "maximum": 4161.864650974025,
  "mean": 255.5219867361225,
  "total": 8323117.673955718,
  "missing_previous_week": 1649,
  "negative_predictions_clipped": 0
}
     id  num_orders
1028232  139.306850
1127204  129.095804
1212707  111.734167
1082698   59.549762
1400926   61.082479


## Limitations and purchasing support
Forecasts can provide a center-meal demand starting point for weekly purchasing review. Translate orders to ingredients only with recipes, lead times, shelf life, on-hand stock and service goals; these data are unavailable, so no exact purchase or safety-stock quantity is claimed. Compare forecasts with actuals weekly using MAE and bias, and monitor stockouts/waste once available.

Ten-week recursive errors can compound; training uses actual histories while future weeks use predicted histories. Missing series observations are not evidence of zero demand. Planning prices/promotions must be known; test prices can exceed training ranges. Unseen categories are encoded safely but may forecast poorly. Clipping prevents negative orders but adds no uncertainty interval. Test has no target, so no test MAE is reported. Center-meal forecasts support managerial review rather than automatic replenishment.

In [6]:
from docs.validate_phase10 import main as validate_phase10
validate_phase10()

{
  "status": "passed",
  "git_ignore_probes": 4,
  "independent_mae": {
    "Previous-week baseline": 119.61305261875019,
    "Linear Regression": 115.68906832568234,
    "Random Forest": 80.48260829770284
  },
  "fit_rows": 423727,
  "holdout_rows": 32821,
  "baseline_recursion": "independently reconstructed",
  "test_ids_and_order": "exact match",
  "test_predictions": 32573,
  "finite_nonnegative": true,
  "current_future_label_poisoning": "identical features",
  "raw_and_prepared_hashes": "unchanged",
  "evaluation_coverage": "all holdout rows; none excluded"
}
